# Read the description of the current state

Read the label and a domain field directly from the selected status object.


## Declare the model

Run these definitions first. This notebook is independent of the other examples.


In [1]:
from pydantic import Field
from aoa.action_machine.domain import BaseStateEntity, Lifecycle, StateVocabulary


class OrderStateEntity(BaseStateEntity):
    """Describe one possible order status."""

    requires_receipt: bool = Field(default=False, description="Whether a receipt is needed")


class OrderLifecycle(Lifecycle, StateVocabulary):
    """Collect statuses and declare their allowed transition."""

    DRAFT = OrderStateEntity(code="draft", display_name="Draft")
    PAID = OrderStateEntity(code="paid", display_name="Payment received", requires_receipt=True)
    _template = Lifecycle().state(DRAFT).to(PAID).initial().state(PAID).final()


## Run the case


In [2]:
row = {"id": "ord-1", "status": "paid"}
current = OrderLifecycle(row["status"])
status = current.current_status
assert status is OrderLifecycle.PAID
assert current.current_state == status.code
assert status.requires_receipt is True
assert OrderLifecycle.entity_type is OrderStateEntity
assert tuple(OrderLifecycle.by_code(code) for code in OrderLifecycle.codes()) == OrderLifecycle.members
print("codes:", OrderLifecycle.codes())
print("current code:", current.current_state)
print("label:", status.display_name)
print("requires receipt:", status.requires_receipt)
print("declared object:", status is OrderLifecycle.PAID)


codes: ['draft', 'paid']
current code: paid
label: Payment received
requires receipt: True
declared object: True


The constructor takes the code from the row and stores a reference to the declared status object. current_status returns that exact object; current_state reads its code without storing a second current code. The requires_receipt field is domain data, not an instruction to issue a receipt. StateVocabulary.by_code remains useful when looking up an arbitrary code rather than the current state.
